# Supplementary Results 2 — Systematic colocalisation

Overlaps between 95% credible sets that share at least one variant, and how many of them
colocalise. Significance follows the threshold the manuscript uses throughout: eCAVIAR
CLPP >= 0.01, COLOC H4 >= 0.8.

The section makes claims on two different universes, and the notebook follows that split:

- the **share of all overlaps** that colocalise is over every tested overlap in the release, with no
  restriction on either side;
- the **credible-set counts** are over the qualifying credible sets only.

Numbers are written to `results/sr02_colocalisation.json`.

**Provenance.** No surviving notebook computes these counts; the two released colocalisation
tables carry everything the qualifying-credible-set paragraph claims, so those numbers are computed
here directly from them. The two overlap percentages come from Supplementary Table 11, which is a
hand-made sheet shipped as a static asset
(`chapters/06-supplementary-tables/assets/ST11_-_coloc_overlap.xlsx`) whose definition could not be
recovered — see the precomputed table below.

In [1]:
import pandas as pd
from gentropy.common.session import Session
from pyspark.sql import functions as f

from manuscript_methods import paper

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})
numbers = {}

CLPP_THRESHOLD = 0.01
H4_THRESHOLD = 0.8

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/20 13:41:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Precomputed numbers

The two percentages in the first paragraph of the section are **precomputed**: they are the row
totals of Supplementary Table 11's subtable 1, a hand-made sheet whose overlap universe is not the
released colocalisation tables and whose recipe was never recovered. Summing that sheet gives
41,398,927 / 61,484,864 = 67.3% for eCAVIAR and 24,536,009 / 31,167,732 = 78.7% for COLOC, which is
where the published 67% and 79% come from. The same quantities over the release are computed below
and give 68.0% and 79.3% — so the COLOC claim agrees only because both readings round to 79.

Subtable 2 of the same sheet, by contrast, **does** agree with this pipeline: its `All` column is
285,229 credible sets and 14,026 genes, the values computed further down. The two subtables were
evidently made against different data.

`tools/check_numbers.py` reports S2.01 and S2.02 as PRECOMPUTED rather than as failures.

In [2]:
PRECOMPUTED = pd.DataFrame(
    [
        {
            "id": "S2.01",
            "claim": "67% of all overlaps significant by eCAVIAR",
            "value": 67.0,
            "why it is not recomputed": "the published share is the total of Supplementary Table 11 "
            "subtable 1 (41,398,927 / 61,484,864 = 67.3%); the release holds 75,407,692 tested "
            "eCAVIAR overlaps, 23% more, and gives 68.0%",
            "input that would close it": "the overlap set ST11 subtable 1 was built on — an earlier "
            "release or an unrecorded filter; not identified",
        },
        {
            "id": "S2.02",
            "claim": "79% of all overlaps significant by COLOC",
            "value": 79.0,
            "why it is not recomputed": "same sheet (24,536,009 / 31,167,732 = 78.7%) against "
            "38,561,709 overlaps and 79.3% in the release; the claim survives rounding either way",
            "input that would close it": "as above",
        },
    ]
)
PRECOMPUTED

,id,claim,value,why it is not recomputed,input that would close it
0,S2.01,67% of all overlaps significant by eCAVIAR,67.0,the published share is the total of Supplement...,the overlap set ST11 subtable 1 was built on —...
1,S2.02,79% of all overlaps significant by COLOC,79.0,"same sheet (24,536,009 / 31,167,732 = 78.7%) a...",as above


## Overlaps and the share that colocalise

Every row of each table is one tested overlap between two credible sets.

In [3]:
ecaviar = session.spark.read.parquet(paper.release("colocalisation_ecaviar")).cache()
coloc = session.spark.read.parquet(paper.release("colocalisation_coloc")).cache()

n_ecaviar = ecaviar.count()
n_coloc = coloc.count()
sig_ecaviar = ecaviar.filter(f.col("clpp") >= CLPP_THRESHOLD).count()
sig_coloc = coloc.filter(f.col("h4") >= H4_THRESHOLD).count()

numbers["S2.01"] = round(100 * sig_ecaviar / n_ecaviar, 1)
numbers["S2.02"] = round(100 * sig_coloc / n_coloc, 1)
print(f"eCAVIAR: {sig_ecaviar:,} of {n_ecaviar:,} overlaps significant ({numbers['S2.01']}%)")
print(f"COLOC:   {sig_coloc:,} of {n_coloc:,} overlaps significant ({numbers['S2.02']}%)")

# The published 67% is a point below what >= 0.01 gives, and the strict inequality of the prose
# ("max CLPP > 0.01") gives the identical count, so the threshold is not the cause: the published
# share is ST11 subtable 1's total over a smaller overlap set (see PRECOMPUTED above).
strict = ecaviar.filter(f.col("clpp") > CLPP_THRESHOLD).count()
print(f"eCAVIAR with a strict CLPP > 0.01: {strict:,} ({100 * strict / n_ecaviar:.1f}%)")

eCAVIAR: 51,261,361 of 75,407,692 overlaps significant (68.0%)
COLOC:   30,597,242 of 38,561,709 overlaps significant (79.3%)


eCAVIAR with a strict CLPP > 0.01: 51,261,361 (68.0%)


In [4]:
# Split by what sits on the right, so the GWAS-vs-GWAS and GWAS-vs-molQTL halves are visible.
overlap_split = (
    ecaviar.groupBy("rightStudyType")
    .agg(
        f.count("*").alias("eCAVIAR overlaps"),
        f.sum(f.when(f.col("clpp") >= CLPP_THRESHOLD, 1).otherwise(0)).alias("eCAVIAR significant"),
    )
    .join(
        coloc.groupBy("rightStudyType").agg(
            f.count("*").alias("COLOC overlaps"),
            f.sum(f.when(f.col("h4") >= H4_THRESHOLD, 1).otherwise(0)).alias("COLOC significant"),
        ),
        "rightStudyType",
        "outer",
    )
    .toPandas()
    .set_index("rightStudyType")
    .sort_values("eCAVIAR overlaps", ascending=False)
)
overlap_split

,eCAVIAR overlaps,eCAVIAR significant,COLOC overlaps,COLOC significant
rightStudyType,,,,
gwas,48998746,39488922,22858410,20853875
eqtl,14787833,5697472,8785076,5039073
tuqtl,4318623,1773479,2482212,1547066
pqtl,3564301,3027099,2208171,2044570
sqtl,2362154,985867,1332416,835766
sceqtl,1376035,288522,895424,276892


## Qualifying credible sets that colocalise with a molecular QTL

The left side of a colocalisation row is the credible set the overlap is reported for. A qualifying
credible set counts once no matter how many molQTL credible sets it colocalises with, and either
method can supply the evidence.

In [5]:
qualifying_cs = (
    session.spark.read.parquet(paper.derived("qualifying_credible_sets"))
    .select("studyLocusId")
    .union(session.spark.read.parquet(paper.derived("qualifying_measurement_credible_sets")).select("studyLocusId"))
    .distinct()
    .cache()
)
n_qualifying = qualifying_cs.count()
print(f"qualifying credible sets: {n_qualifying:,}")

significant = (
    ecaviar.filter(f.col("clpp") >= CLPP_THRESHOLD)
    .select("leftStudyLocusId", "rightStudyLocusId", "rightStudyType")
    .union(coloc.filter(f.col("h4") >= H4_THRESHOLD).select("leftStudyLocusId", "rightStudyLocusId", "rightStudyType"))
    .distinct()
    .filter(f.col("rightStudyType") != "gwas")
    .cache()
)
print(f"significant GWAS-molQTL colocalisations (either method): {significant.count():,}")

qualifying credible sets: 520,975


significant GWAS-molQTL colocalisations (either method): 14,533,915


In [6]:
# The molQTL side, annotated with its gene, whether the association is trans, and the gene biotype.
cs = session.spark.read.parquet(paper.release("credible_set")).select(
    f.col("studyLocusId").alias("rightStudyLocusId"), "studyId", "isTransQtl"
)
si = session.spark.read.parquet(paper.release("study")).select("studyId", "geneId")
target = session.spark.read.parquet(paper.release("target")).select(f.col("id").alias("geneId"), f.col("biotype"))

annotated = (
    significant.join(cs, "rightStudyLocusId", "left")
    .join(si, "studyId", "left")
    .join(target, "geneId", "left")
    .join(qualifying_cs.withColumnRenamed("studyLocusId", "leftStudyLocusId"), "leftStudyLocusId", "inner")
    .cache()
)
print(f"colocalisations whose left credible set qualifies: {annotated.count():,}")

colocalisations whose left credible set qualifies: 10,334,101


In [7]:
def qualifying_with(condition, label):
    """Qualifying credible sets carrying at least one colocalisation of the given kind."""
    rows = annotated if condition is None else annotated.filter(condition)
    n = rows.select("leftStudyLocusId").distinct().count()
    return {"evidence": label, "credible sets": n, "% of qualifying": round(100 * n / n_qualifying, 0)}


protein_coding = f.col("biotype") == "protein_coding"
not_trans = ~f.coalesce(f.col("isTransQtl"), f.lit(False))

summary = pd.DataFrame(
    [
        qualifying_with(None, "any molQTL"),
        qualifying_with(not_trans, "any molQTL, excluding trans-pQTL"),
        qualifying_with(protein_coding, "protein-coding gene molQTL"),
        qualifying_with(protein_coding & not_trans, "protein-coding gene molQTL, excluding trans-pQTL"),
    ]
)
numbers["S2.03"] = int(summary.loc[0, "credible sets"])
numbers["S2.04"] = float(summary.loc[0, "% of qualifying"])
numbers["S2.05"] = int(summary.loc[1, "credible sets"])
numbers["S2.06"] = float(summary.loc[1, "% of qualifying"])
# The published 285,229 sits below the 302,264 of the trans-excluded set, so the protein-coding
# claim is nested inside it rather than being a separate filter on the full set.
numbers["S2.07"] = int(summary.loc[3, "credible sets"])
numbers["S2.08"] = float(summary.loc[3, "% of qualifying"])

numbers["S2.09"] = annotated.filter(protein_coding & not_trans).select("geneId").distinct().count()
print(f"unique protein-coding genes carrying such a colocalisation: {numbers['S2.09']:,}")
summary

unique protein-coding genes carrying such a colocalisation: 14,026


,evidence,credible sets,% of qualifying
0,any molQTL,330584,63.0
1,"any molQTL, excluding trans-pQTL",302264,58.0
2,protein-coding gene molQTL,315913,61.0
3,"protein-coding gene molQTL, excluding trans-pQTL",285229,55.0


## Write the results

In [8]:
print(paper.save_results("sr02_colocalisation", numbers))
pd.Series(numbers).to_frame("computed")

/Users/yt4/Projects/Gentropy-manuscript/results/sr02_colocalisation.json


,computed
S2.01,68.0
S2.02,79.3
S2.03,330584.0
S2.04,63.0
S2.05,302264.0
S2.06,58.0
S2.07,285229.0
S2.08,55.0
S2.09,14026.0
